In [ ]:
import pandas as pd
import psycopg2
import DATABASE_CONFIG

conn = psycopg2.connect(
    dbname=DATABASE_CONFIG.DB_NAME,
    user=DATABASE_CONFIG.DB_USER,
    password=DATABASE_CONFIG.DB_PASSWORD,
    host=DATABASE_CONFIG.DB_HOST,
    port=DATABASE_CONFIG.DB_PORT
)

cursor = conn.cursor()

In [8]:
comando = """
SELECT * FROM regiao_administrativa
WHERE nome = 'Amazônia Legal'
"""
cursor.execute(comando)
resultado = cursor.fetchall()
id = resultado[0][0]
print(id)

1


In [9]:
# Leitura do CSV
df = pd.read_excel('../datasets/EmissaoGasesAmazonia.xlsx', sheet_name='Sem degracação', header=10)


# Seleciona e renomeia as colunas que serão usadas
df = df.rename(columns={
    'Year': 'ano',
    'VR_CO2_1stOrder': 'co2_1a_ordem',
    'VR_CO2_2ndOrder': 'co2_2a_ordem',
    'VR_CO2_2ndOrderFire': 'co2_por_fogo',
    'VR_CO2_2ndOrderDecay': 'co2_por_decaimento',
    'VR_CH42Eq_2ndOrderFire': 'ch4_eq_fogo',
    'VR_N2OEq_2ndOrderFire': 'n2o_eq_fogo',
    'NET_CO2_1stOrder': 'net_co2_1a_ordem',
    'NET_CO2_2ndOrder': 'net_co2_2a_ordem',
    'SV_AreaTotal': 'area_total_secundaria',
    'SV_AreaCleared': 'area_cortada',
    'SV_CO2Emission': 'co2_emitido_secundaria',
    'SV_CO2Absorption': 'co2_absorvido_secundaria',

})

df = df[['ano', 'co2_1a_ordem', 'co2_2a_ordem', 'co2_por_fogo', 'co2_por_decaimento',
           'ch4_eq_fogo', 'n2o_eq_fogo', 'net_co2_2a_ordem',
           'area_total_secundaria', 'area_cortada', 'co2_emitido_secundaria',
           'co2_absorvido_secundaria']]
df['id_regiao_administrativa'] = id


data = list(df.itertuples(index=False, name=None))

In [10]:
from psycopg2.extras import execute_values
comando = """
    INSERT INTO relatorio_emissao_gases_anual
    (ano, co2_1a_ordem, co2_2a_ordem, co2_por_fogo, co2_por_decaimento,
    ch4_eq_fogo, n2o_eq_fogo, net_co2_2a_ordem, area_total_secundaria, area_cortada, co2_emitido_secundaria,
    co2_absorvido_secundaria, id_regiao_administrativa)
    VALUES %s
"""
execute_values(cursor, comando, data)

conn.commit()

In [ ]:
conn.rollback()

In [11]:
cursor.close()
conn.close()